# TrappyTV Examplar notebook

In [ ]:
# Source - https://stackoverflow.com/a/10472712
# Posted by Andrew_1510, modified by community. See post 'Timeline' for change history
# Retrieved 2026-05-04, License - CC BY-SA 4.0

%load_ext autoreload
%autoreload 2


In [ ]:
# Import Packages
import os
import sys
import numpy as np
import pandas as pd

In [ ]:
## Import Bokeh module and TrappyTV class
from bokeh.io import output_notebook
output_notebook()

import sys
sys.path.append(os.path.dirname(os.getcwd()))
from trappytv import CellView


## Import Data

In [ ]:
#datapaths = ["../data/merged_tracks.hd5"]
#datapaths = ["data/2026_01_02/m2_merged_tracks.hd5"]
#datapaths = ["../data/2025_12_30/M4_2025_12_30.hd5"]
datapaths = ["../data/2026_01_02/M2_2026_1_2.hd5"]


In [ ]:
cell = CellView(datapaths[0], compute_speed=True)
cell()

## Do some post-processing

In [ ]:
from analysis.outliers import drop_edge_nans_and_interpolate
cell.add_columns(inputs=["x_unrefined", "y_unrefined"], func=drop_edge_nans_and_interpolate, outputs=["xf", "yf"], max_length=3)
cell()

In [ ]:
from analysis.sg import apply_sg_bf_on_xy, hodrick_prescott_on_xy
cell.add_columns(inputs=["xf", "yf"], func=apply_sg_bf_on_xy, outputs=["xf_11Hz", "yf_11Hz"], inplace=False, window_length=3, polyorder=2)
cell.add_columns(inputs=["xf", "yf"], func=apply_sg_bf_on_xy, outputs=["xf_3Hz", "yf_3Hz"], inplace=False, window_length=7, polyorder=3)
#cell.add_columns(inputs=["xf_3Hz", "yf_3Hz"], func=apply_sg_bf_on_xy, outputs=["xf_long", "yf_long"], inplace=False, window_length=151, polyorder=2)

cell.add_columns(inputs=["xf_3Hz", "yf_3Hz"], func=hodrick_prescott_on_xy, outputs=["xf_long", "yf_long"], inplace=False, lambda_smooth=1e4, 
                 interpolate_if_necessary=True, include_cycles=True)


## Compute speeds
cell()["speedf"] = np.hypot(np.gradient(cell().xf), np.gradient(cell().yf))
cell()["speed_denoise"] = np.hypot(np.gradient(cell().xf_11Hz), np.gradient(cell().yf_11Hz))
cell()["speed_antialiased"] = np.hypot(np.gradient(cell().xf_3Hz), np.gradient(cell().yf_3Hz))
cell()["speed_longterm"] = np.hypot(np.gradient(cell().xf_long), np.gradient(cell().yf_long))



cell()

# Outliers

In [ ]:
%%time
from analysis.outliers import detect_outliers_local
events = detect_outliers_local(cell(), no_tbins=500, ep_col="ep_unrefined")

In [ ]:
print("All events: ", len(events))
print("All events with signal dip: ", len([event for event in events if event["type"] == "signal_dip"]))


# View Data

In [ ]:
## Clip if necessary
cell.dfs["tracks"] = cell.dfs["tracks"][cell.dfs["tracks"].split < 10]

In [ ]:
#%%time
from trappytv import TrappyTV
tv = TrappyTV(cell, width=900, height=900, filtered_columns={"filtered": ["xf", "yf"],
                                                                  "denoised": ["xf_11Hz", "yf_11Hz"],
                                                                  "anti-aliased": ["xf_3Hz", "yf_3Hz"],
                                                                  "long-term": ["xf_long", "yf_long"]})
tv.view_split(split_no=0)    ## Show only a particular split -> render all points
#tv.view_all(sample=10)    ## Show only a particular split -> render all points


In [ ]:
tv